# Cardiac AI V4 - ECG Analyzer
Upload file ECG (.mat + .hea) de phan tich 27 loai benh tim

**Model: 1D Multi-scale CNN + Transformer (20M params) | F1=0.538 | AUC=0.926**

## Cell 1 - Cai thu vien

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'])
print('Done')

## Cell 2 - Imports & Config

In [ ]:
import os, json, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
warnings.filterwarnings('ignore')

import wfdb
import torch
import torch.nn as nn

NUM_LEADS    = 12
SEQ_LEN      = 4096
DROPOUT      = 0.15
MS_CHANNELS  = [128, 256, 512, 512]
TRANS_DIM    = 384
TRANS_HEADS  = 8
TRANS_LAYERS = 8
TRANS_FF_DIM = 1536
NUM_CLASSES  = 27

LEAD_NAMES = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']

IDX_TO_NAME = {
    0:'IAVB', 1:'AF', 2:'AFL', 3:'Brady', 4:'CRBBB', 5:'IRBBB',
    6:'LAnFB', 7:'LAD', 8:'LBBB', 9:'LQRSV', 10:'NSIVCB', 11:'PR',
    12:'PAC', 13:'PVC', 14:'LPR', 15:'LQT', 16:'QAb', 17:'RAD',
    18:'RBBB', 19:'SA', 20:'SB', 21:'SNR', 22:'STach', 23:'SVPB',
    24:'TAb', 25:'TInv', 26:'VPB'
}

IDX_TO_FULLNAME = {
    0:'1st Degree AV Block', 1:'Atrial Fibrillation', 2:'Atrial Flutter',
    3:'Bradycardia', 4:'Complete Right Bundle Branch Block',
    5:'Incomplete Right Bundle Branch Block', 6:'Left Anterior Fascicular Block',
    7:'Left Axis Deviation', 8:'Left Bundle Branch Block',
    9:'Low QRS Voltages', 10:'Nonspecific Intraventricular Conduction Disturbance',
    11:'PR Interval', 12:'Premature Atrial Contraction',
    13:'Premature Ventricular Contraction', 14:'Prolonged PR Interval',
    15:'Prolonged QT Interval', 16:'Q Wave Abnormal', 17:'Right Axis Deviation',
    18:'Right Bundle Branch Block', 19:'Sinus Arrhythmia', 20:'Sinus Bradycardia',
    21:'Sinus Rhythm', 22:'Sinus Tachycardia', 23:'Supraventricular Premature Beats',
    24:'T-wave Abnormal', 25:'T-wave Inversion', 26:'Ventricular Premature Beats'
}

IDX_TO_SEVERITY = {
    0:'warning', 1:'danger', 2:'danger', 3:'warning', 4:'warning', 5:'info',
    6:'info', 7:'info', 8:'danger', 9:'info', 10:'info', 11:'info',
    12:'warning', 13:'warning', 14:'warning', 15:'warning', 16:'warning',
    17:'info', 18:'warning', 19:'info', 20:'info', 21:'normal',
    22:'warning', 23:'warning', 24:'warning', 25:'warning', 26:'warning'
}

SEVERITY_COLOR = {
    'normal':'#22c55e', 'info':'#3b82f6',
    'warning':'#f59e0b', 'danger':'#ef4444'
}

# ── SUA PATH NAY THEO DATASET CUA BAN ──────────────────────────
MODEL_PATH     = '/kaggle/input/datasets/tuyenngoc12/cardiac-ai-v3-model/cardiac_v4_best.pth'
THRESHOLD_PATH = '/kaggle/input/datasets/tuyenngoc12/cardiac-ai-v3-model/best_thresholds_v4.json'
# ───────────────────────────────────────────────────────────────

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Config OK')

## Cell 3 - Load Model V4

In [ ]:
class MultiScaleConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=2):
        super().__init__()
        assert out_ch % 4 == 0
        b = out_ch // 4
        self.branch_small = nn.Sequential(
            nn.Conv1d(in_ch, b, 3,  padding=1,  bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.branch_mid = nn.Sequential(
            nn.Conv1d(in_ch, b, 9,  padding=4,  bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.branch_large = nn.Sequential(
            nn.Conv1d(in_ch, b, 19, padding=9,  bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.branch_pool = nn.Sequential(
            nn.AvgPool1d(3, stride=1, padding=1),
            nn.Conv1d(in_ch, b, 1, bias=False),
            nn.GroupNorm(min(8,b), b), nn.GELU())
        self.fusion = nn.Sequential(
            nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(min(8,out_ch), out_ch), nn.GELU())
        self.pool     = nn.MaxPool1d(pool)
        self.shortcut = (nn.Sequential(
                            nn.Conv1d(in_ch, out_ch, 1, bias=False),
                            nn.GroupNorm(min(8,out_ch), out_ch))
                         if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        out = self.fusion(torch.cat([
            self.branch_small(x), self.branch_mid(x),
            self.branch_large(x), self.branch_pool(x)], dim=1))
        return self.pool(out + self.shortcut(x))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=DROPOUT):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0)/d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class ECGModelV4(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        channels = [NUM_LEADS] + MS_CHANNELS
        self.cnn = nn.Sequential(*[
            MultiScaleConvBlock(channels[i], channels[i+1])
            for i in range(len(MS_CHANNELS))])
        self.proj    = nn.Sequential(
            nn.Linear(MS_CHANNELS[-1], TRANS_DIM),
            nn.LayerNorm(TRANS_DIM))
        self.pos_enc = PositionalEncoding(TRANS_DIM, max_len=1024)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=TRANS_DIM, nhead=TRANS_HEADS, dim_feedforward=TRANS_FF_DIM,
            dropout=DROPOUT, activation='gelu', batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=TRANS_LAYERS)
        self.head = nn.Sequential(
            nn.LayerNorm(TRANS_DIM), nn.Dropout(DROPOUT),
            nn.Linear(TRANS_DIM, 512), nn.GELU(),
            nn.Dropout(DROPOUT/2), nn.Linear(512, num_classes))
    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        x = self.proj(x)
        x = self.pos_enc(x)
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.head(x)

import torch
model = ECGModelV4().to(device)
ckpt  = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['state'])
model.eval()
print(f'Model V4 loaded! epoch={ckpt["epoch"]} val_F1={ckpt["score"]:.4f}')

with open(THRESHOLD_PATH) as f:
    thresh_dict = json.load(f)
THRESHOLDS = np.array([thresh_dict[str(i)] for i in range(NUM_CLASSES)])
print('Thresholds loaded!')
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

## Cell 4 - Ham Inference & Ve bieu do

In [ ]:
def load_ecg(mat_path):
    try:
        rec = wfdb.rdrecord(mat_path.replace('.mat',''))
        sig = rec.p_signal.astype(np.float32)
        fs  = rec.fs
    except Exception as e:
        print(f'Loi: {e}')
        return None, None, None
    mu  = sig.mean(0, keepdims=True)
    std = sig.std(0,  keepdims=True) + 1e-8
    sig_norm = (sig - mu) / std
    T = sig_norm.shape[0]
    if T >= SEQ_LEN:
        start = (T - SEQ_LEN) // 2
        sig_crop = sig_norm[start:start+SEQ_LEN]
    else:
        sig_crop = np.vstack([sig_norm,
                   np.zeros((SEQ_LEN-T, NUM_LEADS), dtype=np.float32)])
    return sig, sig_crop, fs

def predict_tta(sig_crop, n_tta=5):
    import random
    x0 = torch.from_numpy(sig_crop.T).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(x0)).squeeze().cpu().numpy()

    for _ in range(n_tta):
        aug = sig_crop.copy()
        aug += np.random.normal(0, 0.03, aug.shape).astype(np.float32)
        aug *= np.random.uniform(0.85, 1.15, (1, NUM_LEADS)).astype(np.float32)
        x_aug = torch.from_numpy(aug.T).unsqueeze(0).to(device)
        with torch.no_grad():
            probs += torch.sigmoid(model(x_aug)).squeeze().cpu().numpy()

    probs /= (n_tta + 1)
    preds  = (probs >= THRESHOLDS).astype(int)
    return probs, preds

def plot_ecg(sig_raw, fs=500, title='ECG'):
    n_leads = min(sig_raw.shape[1], 12)
    t = np.arange(min(sig_raw.shape[0], fs*10)) / fs
    sig_plot = sig_raw[:len(t)]
    fig, axes = plt.subplots(n_leads, 1, figsize=(18, n_leads*1.2), facecolor='#0f172a')
    fig.suptitle(title, color='white', fontsize=13, fontweight='bold')
    for i in range(n_leads):
        ax = axes[i]
        ax.set_facecolor('#0f172a')
        ax.plot(t, sig_plot[:, i], color='#22d3ee', linewidth=0.7, alpha=0.9)
        ax.set_ylabel(LEAD_NAMES[i], color='#94a3b8', fontsize=8,
                      rotation=0, labelpad=22, va='center')
        ax.set_xlim(t[0], t[-1])
        ax.tick_params(colors='#475569', labelsize=7)
        ax.grid(True, alpha=0.12, color='#334155', linewidth=0.4)
        for sp in ax.spines.values(): sp.set_color('#1e293b')
    axes[-1].set_xlabel('Time (s)', color='#94a3b8', fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close()

def plot_probs(probs, preds):
    detected   = [(i, probs[i]) for i in range(NUM_CLASSES) if preds[i]==1]
    top_others = sorted([(i, probs[i]) for i in range(NUM_CLASSES) if preds[i]==0],
                         key=lambda x:-x[1])[:8]
    all_items  = sorted(detected + top_others, key=lambda x:-x[1])
    fig, ax = plt.subplots(figsize=(12, max(5, len(all_items)*0.5)), facecolor='#0f172a')
    ax.set_facecolor('#0f172a')
    for idx, (ci, prob) in enumerate(all_items):
        is_det = preds[ci]==1
        color  = SEVERITY_COLOR[IDX_TO_SEVERITY.get(ci,'info')] if is_det else '#334155'
        ax.barh(idx, prob, color=color, alpha=1.0 if is_det else 0.5,
                height=0.65, edgecolor='#1e293b', linewidth=0.5)
        ax.axvline(x=THRESHOLDS[ci], ymin=idx/len(all_items),
                   ymax=(idx+1)/len(all_items),
                   color='white', linewidth=1.0, alpha=0.5, linestyle='--')
        label = f'{IDX_TO_NAME[ci]} ({prob:.2f})' + (' DETECTED' if is_det else '')
        ax.text(prob+0.01, idx, label,
                color='white' if is_det else '#64748b',
                va='center', fontsize=8.5,
                fontweight='bold' if is_det else 'normal')
    ax.set_yticks(range(len(all_items)))
    ax.set_yticklabels([IDX_TO_NAME[ci] for ci,_ in all_items],
                        color='#94a3b8', fontsize=8)
    ax.set_xlim(0, 1.35)
    ax.set_xlabel('Probability', color='#94a3b8')
    ax.set_title('Prediction Probabilities (-- = threshold)', color='white', fontsize=12)
    ax.tick_params(colors='#475569')
    for sp in ax.spines.values(): sp.set_color('#1e293b')
    ax.grid(axis='x', alpha=0.12, color='#334155')
    legend = [mpatches.Patch(color=c, label=l) for l,c in [
        ('Normal','#22c55e'),('Info','#3b82f6'),
        ('Warning','#f59e0b'),('Danger','#ef4444')]]
    ax.legend(handles=legend, loc='lower right',
              facecolor='#1e293b', labelcolor='white', fontsize=8)
    plt.tight_layout()
    plt.show()
    plt.close()

def show_results_html(probs, preds, record_name=''):
    detected = sorted([(i, probs[i]) for i in range(NUM_CLASSES) if preds[i]==1],
                       key=lambda x:-x[1])
    badge_map = {'normal':'BINH THUONG','info':'THONG TIN',
                 'warning':'CANH BAO','danger':'NGUY HIEM'}
    rows = ''
    if not detected:
        rows = '<div style="color:#22c55e;font-size:16px;padding:10px">Khong phat hien bat thuong</div>'
    else:
        for ci, prob in detected:
            sev   = IDX_TO_SEVERITY.get(ci,'info')
            color = SEVERITY_COLOR[sev]
            badge = badge_map.get(sev,'')
            rows += (
                f'<div style="background:#1e293b;border-left:4px solid {color};'
                f'padding:10px 14px;margin:6px 0;border-radius:6px">'
                f'<span style="color:{color};font-weight:bold;font-size:15px">{IDX_TO_NAME[ci]}</span>'
                f'<span style="background:{color};color:white;padding:2px 8px;'
                f'border-radius:4px;font-size:11px;margin-left:8px">{badge}</span><br>'
                f'<span style="color:#94a3b8;font-size:13px">{IDX_TO_FULLNAME.get(ci,"")}</span><br>'
                f'<div style="margin-top:6px">'
                f'<div style="background:#334155;border-radius:4px;height:8px;width:100%">'
                f'<div style="background:{color};border-radius:4px;height:8px;width:{prob*100:.1f}%"></div></div>'
                f'<span style="color:#64748b;font-size:12px">'
                f'Xac suat: {prob:.4f} | Nguong: {THRESHOLDS[ci]:.2f}</span>'
                f'</div></div>'
            )
    n_det = len(detected)
    status_color = '#ef4444' if any(IDX_TO_SEVERITY.get(ci,'')=='danger' for ci,_ in detected)                    else '#f59e0b' if n_det > 0 else '#22c55e'
    status_text  = f'Phat hien {n_det} bat thuong' if n_det > 0 else 'Binh thuong'
    html = (
        f'<div style="font-family:monospace;background:#0f172a;padding:20px;'
        f'border-radius:12px;color:white;margin:10px 0">'
        f'<h2 style="color:#22d3ee;margin-bottom:8px">Cardiac AI V4 - Ket qua ECG</h2>'
        f'<div style="color:#64748b;font-size:12px;margin-bottom:4px">{record_name}</div>'
        f'<div style="color:#475569;font-size:11px;margin-bottom:12px">'
        f'Model: Multi-scale CNN + Transformer (20M) | F1=0.538 | AUC=0.926 | TTA=5x</div>'
        f'<div style="color:{status_color};font-size:16px;font-weight:bold;margin-bottom:12px">'
        f'{status_text}</div>'
        f'{rows}</div>'
    )
    display(HTML(html))

print('Ham inference V4 OK')

## Cell 5 - Giao dien Upload & Test

In [ ]:
header_html = (
    '<div style="font-family:monospace;background:#0f172a;padding:20px;'
    'border-radius:12px;color:white;margin-bottom:10px">'
    '<h2 style="color:#22d3ee;margin:0 0 8px 0">Cardiac AI V4 - ECG Analyzer</h2>'
    '<p style="color:#94a3b8;margin:0">Upload file ECG (.mat va .hea) de phan tich 27 loai benh tim</p>'
    '<p style="color:#475569;font-size:12px;margin:4px 0 0 0">'
    'Multi-scale CNN + Transformer (20M) | F1=0.538 | AUC=0.926 | TTA 5x</p>'
    '</div>'
)
display(HTML(header_html))

upload_mat  = widgets.FileUpload(accept='.mat', multiple=False, description='Upload .mat')
upload_hea  = widgets.FileUpload(accept='.hea', multiple=False, description='Upload .hea')
btn_analyze = widgets.Button(description='Phan tich ECG', button_style='primary',
                              layout=widgets.Layout(width='180px', height='38px'))
out = widgets.Output()

def on_click(b):
    with out:
        clear_output()
        if not upload_mat.value or not upload_hea.value:
            display(HTML('<div style="color:#ef4444;font-family:monospace">Vui long upload ca 2 file .mat va .hea!</div>'))
            return

        val_mat = upload_mat.value
        val_hea = upload_hea.value
        if isinstance(val_mat, tuple):
            mat_name = val_mat[0]['name']; mat_data = val_mat[0]['content']
            hea_name = val_hea[0]['name']; hea_data = val_hea[0]['content']
        else:
            mat_name = list(val_mat.keys())[0]; mat_data = val_mat[mat_name]['content']
            hea_name = list(val_hea.keys())[0]; hea_data = val_hea[hea_name]['content']

        mat_path = f'/kaggle/working/tmp_{mat_name}'
        hea_path = f'/kaggle/working/tmp_{hea_name}'
        with open(mat_path, 'wb') as f: f.write(mat_data)
        with open(hea_path, 'wb') as f: f.write(hea_data)

        record_name = mat_name.replace('.mat','')
        display(HTML(f'<div style="color:#22d3ee;font-family:monospace">Dang xu ly TTA 5x: {record_name}...</div>'))

        sig_raw, sig_crop, fs = load_ecg(mat_path)
        if sig_raw is None:
            display(HTML('<div style="color:#ef4444">Loi doc file ECG!</div>'))
            return

        probs, preds = predict_tta(sig_crop, n_tta=5)
        show_results_html(probs, preds, record_name)

        display(HTML('<h3 style="color:#22d3ee;font-family:monospace">Tin hieu ECG (10 giay dau)</h3>'))
        plot_ecg(sig_raw, fs, title=f'ECG - {record_name}')

        display(HTML('<h3 style="color:#22d3ee;font-family:monospace">Xac suat du doan tung benh</h3>'))
        plot_probs(probs, preds)

        os.remove(mat_path)
        os.remove(hea_path)

btn_analyze.on_click(on_click)
display(widgets.VBox([
    widgets.HBox([upload_mat, upload_hea]),
    btn_analyze,
    out
]))